<a href="https://colab.research.google.com/github/sohailpayami2023/digital-signal-processing-python/blob/main/notebooks/00_python_foundations/06_pandas_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas Basics for DSP and ML Engineers

Pandas provides a **DataFrame** — a labelled table of
data, like a MATLAB table or an Excel spreadsheet in
Python. It is the standard way to organise simulation
results, load measurement data, and feed data into
machine learning pipelines.

**Topics covered**
1. Series and DataFrame
2. Creating DataFrames
3. Indexing and slicing
4. Adding and filtering data
5. Reading and writing files (CSV, .mat, .npy)
6. Groupby and aggregation
7. Plotting from a DataFrame
8. Practical example — BER sweep results
9. MATLAB → Python quick reference


# 1. Series and DataFrame

A **Series** is a labelled 1-D array — one column.
A **DataFrame** is a labelled 2-D table — rows and columns,
each column being a Series.

| Pandas | MATLAB equivalent |
|--------|-------------------|
| `Series` | Named vector / column vector |
| `DataFrame` | `table` |
| Column name | Variable name in a table |
| Index (row labels) | Row names / `RowNames` |

Unlike MATLAB tables, Pandas DataFrames can hold
mixed types per column and have a rich set of
filtering, grouping, and aggregation methods.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize' : (5, 3),
    'figure.dpi'     : 120,
    'axes.grid'      : True,
    'grid.alpha'     : 0.4,
})

# Series — a labelled 1-D array
ber = pd.Series([0.5, 0.3, 0.1, 0.01, 0.001],
                name='BER')
print('Series:')
print(ber)
print('dtype:', ber.dtype)

# Access by position (like numpy)
print('\nber[0]:', ber.iloc[0])
print('ber[2:4]:', ber.iloc[2:4].values)


# 2. Creating DataFrames

The most common way is from a **dictionary** where
each key is a column name and each value is a list
or array of column data.

MATLAB equivalent: `table(col1, col2, ...,
VariableNames={'name1','name2',...})`


In [ ]:
# From a dictionary — most common approach
# MATLAB: table(snr_db, ber_awgn, ber_fading,
#          VariableNames={'SNR_dB','BER_AWGN','BER_Fading'})
df = pd.DataFrame({
    'SNR_dB'    : [0, 5, 10, 15, 20],
    'BER_AWGN'  : [0.50, 0.30, 0.10, 0.01, 0.001],
    'BER_Fading': [0.50, 0.40, 0.25, 0.08, 0.015],
})
print(df)

print('\nShape    :', df.shape)   # (rows, cols)
print('Columns  :', df.columns.tolist())
print('Dtypes:\n', df.dtypes)


In [ ]:
# From numpy arrays — useful after a simulation loop
snr   = np.arange(0, 25, 5)
ber   = 0.5 * np.exp(-10**(snr/10))   # simplified
scheme = ['BPSK'] * len(snr)

df2 = pd.DataFrame({
    'SNR_dB'    : snr,
    'BER'       : ber,
    'Modulation': scheme,
})
print(df2)


# 3. Indexing and Slicing

| Operation | Pandas | MATLAB table |
|-----------|--------|-------------|
| Select column | `df["col"]` | `T.col` or `T.("col")` |
| Select row by position | `df.iloc[i]` | `T(i,:)` |
| Select row by label | `df.loc[label]` | `T("label",:)` |
| Slice rows | `df.iloc[2:5]` | `T(3:5,:)` |
| Boolean filter | `df[df["col"] > x]` | `T(T.col > x, :)` |


In [ ]:
# Select a single column — returns a Series
print(df['SNR_dB'])        # MATLAB: df.SNR_dB

# Select multiple columns — returns a DataFrame
print(df[['SNR_dB','BER_AWGN']])

# Row by integer position — MATLAB: df(3,:)
print('\nRow 2 (0-indexed):')
print(df.iloc[2])

# Slice rows — MATLAB: df(2:4,:)
print('\nRows 1-3:')
print(df.iloc[1:4])

# Boolean filter — MATLAB: df(df.SNR_dB >= 10,:)
print('\nSNR >= 10 dB:')
print(df[df['SNR_dB'] >= 10])


# 4. Adding and Filtering Data


In [ ]:
# Add a computed column
# MATLAB: df.SNR_lin = 10.^(df.SNR_dB/10)
df['SNR_lin'] = 10 ** (df['SNR_dB'] / 10)

# Add a categorical column
df['Channel'] = 'AWGN'
print(df)

# Drop a column — MATLAB: removevars(df, "SNR_lin")
df_clean = df.drop(columns=['SNR_lin','Channel'])
print(df_clean)

# Rename columns — MATLAB: df.Properties.VariableNames
df_renamed = df_clean.rename(columns={
    'BER_AWGN'  : 'BER (AWGN)',
    'BER_Fading': 'BER (Fading)',
})
print(df_renamed.columns.tolist())


# 5. Reading and Writing Files

## CSV files
The most portable format for simulation results.
MATLAB: `readtable("f.csv")` / `writetable(T,"f.csv")`

## NumPy `.npy` / `.npz` files
Fast binary format for numpy arrays.
MATLAB: no direct equivalent (closest: `.mat`)

## MATLAB `.mat` files
`scipy.io.loadmat()` reads `.mat` files — very useful
if you have existing MATLAB simulation data.


In [ ]:
import tempfile, os

# ── CSV ─────────────────────────────────────────────────
# Save to CSV — MATLAB: writetable(df, "results.csv")
csv_path = tempfile.mktemp(suffix='.csv')
df.to_csv(csv_path, index=False)
print('Saved CSV to:', csv_path)

# Load from CSV — MATLAB: T = readtable("results.csv")
df_loaded = pd.read_csv(csv_path)
print('Loaded shape:', df_loaded.shape)
print(df_loaded.head(3))

os.remove(csv_path)


In [ ]:
from scipy.io import loadmat, savemat

# ── .npy (numpy binary) ──────────────────────────────────
arr = np.array([1.0, 2.0, 3.0])
npy_path = tempfile.mktemp(suffix='.npy')
# Save — fast, exact precision
np.save(npy_path, arr)
# Load
arr_back = np.load(npy_path)
print('npy round-trip:', arr_back)
os.remove(npy_path)

# ── .mat (MATLAB file) ───────────────────────────────────
# Save a .mat file (MATLAB can open this)
mat_path = tempfile.mktemp(suffix='.mat')
savemat(mat_path, {'snr': np.arange(0,20,5),
                    'ber': np.array([0.5,0.1,0.01,0.001])})

# Load a .mat file — works for MATLAB files up to v7.3
mat = loadmat(mat_path)
# loadmat adds metadata keys starting with '_'
snr = mat['snr'].flatten()
ber = mat['ber'].flatten()
print('snr from .mat:', snr)
print('ber from .mat:', ber)
os.remove(mat_path)


# 6. Groupby and Aggregation

`groupby` splits the DataFrame by the values in one
column, then applies a function to each group.
Ideal for averaging Monte Carlo runs at each SNR point.

MATLAB equivalent: `groupsummary(T, "group_col", "mean")`


In [ ]:
# Simulate multiple Monte Carlo runs at each SNR
np.random.seed(0)
rows = []
for snr in [0, 5, 10, 15]:
    for run in range(5):   # 5 independent runs
        # Simplified BER with random variation
        ber_val = (0.5 * 10**(-snr/10)
                   * np.random.uniform(0.8, 1.2))
        rows.append({
            'SNR_dB': snr,
            'Run'   : run,
            'BER'   : ber_val,
        })

df_runs = pd.DataFrame(rows)
print('All runs:')
print(df_runs.to_string(index=False))


In [ ]:
# Average over runs for each SNR point
# MATLAB: groupsummary(df_runs, "SNR_dB", "mean", "BER")
df_avg = df_runs.groupby('SNR_dB')['BER'].agg(
    mean_ber='mean',
    std_ber='std',
    n_runs='count',
).reset_index()
print('\nAveraged results:')
print(df_avg)


# 7. Plotting from a DataFrame

Pandas has a built-in `.plot()` method that wraps
Matplotlib. For simple plots it is convenient;
for publication-quality work use Matplotlib directly.


In [ ]:
# Quick plot from DataFrame
# MATLAB: semilogy(T.SNR_dB, T.BER_AWGN)
fig, ax = plt.subplots(figsize=(5, 3))

# Pandas plot method — passes kwargs to Matplotlib
df.plot(x='SNR_dB', y=['BER_AWGN','BER_Fading'],
        logy=True, marker='o', ax=ax)
ax.set(xlabel='SNR (dB)', ylabel='BER',
       title='BER vs SNR — Pandas plot')
plt.tight_layout()
plt.show()

# Using averaged results with error bars
fig, ax = plt.subplots(figsize=(5, 3))
ax.errorbar(
    df_avg['SNR_dB'],
    df_avg['mean_ber'],
    yerr=df_avg['std_ber'],
    fmt='o-', capsize=4, label='Mean ± std'
)
ax.set_yscale('log')
ax.set(xlabel='SNR (dB)', ylabel='BER',
       title='Monte Carlo — mean BER with error bars')
ax.legend()
plt.tight_layout()
plt.show()


# 8. Practical Example — Full BER Sweep

A complete workflow: simulate → store in DataFrame
→ save CSV → reload → plot. This is the pattern you
will use in every simulation notebook.


In [ ]:
from scipy.special import erfc

np.random.seed(42)
n_bits     = 50_000
snr_range  = np.arange(0, 13, 2)
modulations = ['BPSK', 'QPSK']

results = []

for mod in modulations:
    # QPSK has same BER as BPSK per bit (Gray coding)
    for snr in snr_range:
        snr_lin  = 10**(snr/10)
        bits     = np.random.randint(0, 2, n_bits)
        tx       = 2*bits - 1
        noise_std = np.sqrt(1/(2*snr_lin))
        rx       = tx + noise_std*np.random.randn(n_bits)
        ber_mc   = np.mean((rx > 0).astype(int) != bits)
        ber_th   = 0.5 * erfc(np.sqrt(snr_lin))
        results.append({
            'Modulation': mod,
            'SNR_dB'    : snr,
            'BER_MC'    : ber_mc,
            'BER_Theory': ber_th,
        })

df_ber = pd.DataFrame(results)
print(df_ber.head(8).to_string(index=False))

# Save and reload
csv_path = tempfile.mktemp(suffix='.csv')
df_ber.to_csv(csv_path, index=False)
df_reload = pd.read_csv(csv_path)
print(f'\nReloaded {len(df_reload)} rows from CSV')
os.remove(csv_path)


In [ ]:
# Plot results for each modulation scheme
fig, ax = plt.subplots(figsize=(5, 3))

for mod, grp in df_ber.groupby('Modulation'):
    ax.semilogy(grp['SNR_dB'], grp['BER_MC'],
                'o--', ms=4, label=f'{mod} MC')
    ax.semilogy(grp['SNR_dB'], grp['BER_Theory'],
                '-', label=f'{mod} Theory')

ax.set(xlabel='SNR (dB)', ylabel='BER',
       title='BER — Monte Carlo vs Theory')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


# 9. MATLAB → Pandas Quick Reference

| Operation | MATLAB | Pandas |
|-----------|--------|--------|
| Create table | `table(a,b)` | `pd.DataFrame({"a":a,"b":b})` |
| Column names | `T.Properties.VariableNames` | `df.columns` |
| Shape | `size(T)` | `df.shape` |
| Select column | `T.col` | `df["col"]` |
| Select row (pos) | `T(i,:)` | `df.iloc[i]` |
| Filter rows | `T(T.col>x,:)` | `df[df["col"]>x]` |
| Add column | `T.new = values` | `df["new"] = values` |
| Drop column | `removevars(T,"col")` | `df.drop(columns=["col"])` |
| Rename column | `T.Properties.VariableNames{i}=...` | `df.rename(columns={...})` |
| Sort | `sortrows(T,"col")` | `df.sort_values("col")` |
| Group + mean | `groupsummary(T,"g","mean","v")` | `df.groupby("g")["v"].mean()` |
| Summary stats | `summary(T)` | `df.describe()` |
| Read CSV | `readtable("f.csv")` | `pd.read_csv("f.csv")` |
| Write CSV | `writetable(T,"f.csv")` | `df.to_csv("f.csv",index=False)` |
| Read .mat | `load("f.mat")` | `scipy.io.loadmat("f.mat")` |
| Quick plot | `plot(T.x, T.y)` | `df.plot(x="x",y="y")` |
| Unique values | `unique(T.col)` | `df["col"].unique()` |
| Value counts | `histc` / `tabulate` | `df["col"].value_counts()` |
| Null check | `ismissing(T)` | `df.isna()` |
| Drop nulls | `rmmissing(T)` | `df.dropna()` |
